# Using the OpenAI SDK

A **large language model** (LLM) is the kind of AI behind chat assistants. You give it text, and it writes text back. In this notebook you'll talk to one from Python instead of from a chat window, which means your own programs can ask questions and use the answers.

Three pieces make that work:

- An **API** is the way one program asks another for something over the internet. Model providers offer an API for sending text to a model and getting its reply.
- The **OpenAI SDK** is a Python library that makes the API easy to use. Its way of talking to models has become a standard, so the same code works with models from many companies.
- **OpenRouter** is a service that passes your request on to the model you choose. One account and one key give you hundreds of models.

## Before you start

You need an OpenRouter **API key**, a secret code that identifies your account. Your teacher may give you one, or you can create your own at openrouter.ai.

Add it on the **Settings** page as an environment variable named `OPENROUTER_API_KEY`. [[503B5721-E671-4BE2-992C-2B8782BC43F8|Environment Variables]] walks through the steps, and explains why a key should never be typed into a notebook.

# Creating a Client

Everything starts with a **client**, the object that sends your requests. It needs two things:

- `base_url` is the address to send requests to. Here that is OpenRouter.
- `api_key` is your key, read from the environment variable.

Run this cell first. Every other cell in the notebook uses the `client` it creates.

In [ ]:
import openai
import os

api_key = os.getenv('OPENROUTER_API_KEY')
if not api_key:
  raise RuntimeError('Add OPENROUTER_API_KEY on the Settings page, then run this cell again.')

client = openai.OpenAI(
  base_url='https://openrouter.ai/api/v1',
  api_key=api_key
)

print('Client ready.')

# Sending a Message

A request has two main parts:

- `model` is the name of the model to use. On OpenRouter a name looks like `company/model`.
- `messages` is a list of the messages so far. Each message is a dictionary with a `role`, which says who is speaking, and the `content`, which is what they said.

The `user` role is the person asking. The reply comes back inside a response object, and the text is at `response.choices[0].message.content`.

The dropdown picks the model for the rest of the notebook. Try the same question with a different one.

> **Tip:** An `AuthenticationError` means the key is wrong or has been switched off. Check it on the Settings page.

In [ ]:
MODEL = "google/gemma-3-27b-it" #@param ["google/gemma-3-27b-it", "openai/gpt-5-nano", "openai/gpt-5.1-chat"]

response = client.chat.completions.create(
  model=MODEL,
  messages=[
    {"role": "user", "content": "Which countries in Europe start with the letter B?"},
  ]
)

print(response.choices[0].message.content)

## Inside the Response

The response carries more than the reply. Two parts are worth knowing:

- `finish_reason` says why the model stopped writing. `stop` means it reached the end of its answer.
- `usage` counts **tokens**. A token is a small chunk of text, often a word or part of a word. Models read and write in tokens, and providers charge by the token, so this is how you see what a request cost.

In [ ]:
print('Answered by:', response.model)
print('Finish reason:', response.choices[0].finish_reason)
print('Tokens in your messages:', response.usage.prompt_tokens)
print('Tokens in the reply:', response.usage.completion_tokens)

# The System Role

A message with the `system` role comes first and tells the model *how to behave*: who it is, who it is talking to, and what style to answer in. The person chatting never sees it, but it shapes every reply.

The cell below asks one question, and you choose the system message. Pick a different persona and run it again to see how much the answer changes.

In [ ]:
PERSONA = "a patient science teacher" #@param ["a patient science teacher", "a pirate captain", "a sports commentator", "a poet who answers in rhyme"]
QUESTION = "Why is the sky blue?" #@param

response = client.chat.completions.create(
  model=MODEL,
  messages=[
    {"role": "system", "content": f"You are {PERSONA}. Answer in three sentences or fewer."},
    {"role": "user", "content": QUESTION},
  ]
)

print(response.choices[0].message.content)

# Having a Conversation

A model has **no memory**. Each request is brand new to it, and it knows only what is in the `messages` list you send.

So to hold a conversation, you keep the list yourself. After each reply, add it to the list with the `assistant` role, then add the next `user` message, and send the *whole* list again.

In the cell below the second question is just "What about Asia?". That only makes sense because the first question and its answer are sent along with it.

In [ ]:
messages = [
  {"role": "system", "content": "You help students with geography. Keep answers short."},
  {"role": "user", "content": "What is the longest river in Africa?"},
]

response = client.chat.completions.create(model=MODEL, messages=messages)
answer = response.choices[0].message.content
print('First answer:', answer)

messages.append({"role": "assistant", "content": answer})
messages.append({"role": "user", "content": "What about Asia?"})

response = client.chat.completions.create(model=MODEL, messages=messages)
print('Second answer:', response.choices[0].message.content)

# Streaming

You may have noticed a wait before each answer appears. A model writes its reply one token at a time, but so far your code has waited for the last token before showing anything.

With `stream=True` the reply arrives as it is written. Instead of one finished response you get a series of **chunks**, each holding the next small piece of text in `chunk.choices[0].delta.content`. A `for` loop prints each piece as it comes in.

`end=''` stops `print` from starting a new line after every piece, so the pieces join up into normal text.

Streaming doesn't make the model any faster. It lets the reader start reading sooner, which is why chat apps use it.

In [ ]:
stream = client.chat.completions.create(
  model=MODEL,
  messages=[
    {"role": "system", "content": "You are a storyteller for teenagers."},
    {"role": "user", "content": "Tell a short story about a robot who learns to paint."},
  ],
  stream=True
)

for chunk in stream:
  if chunk.choices and chunk.choices[0].delta.content:
    print(chunk.choices[0].delta.content, end='', flush=True)

## Keeping the Streamed Text

Printing the pieces shows them, but then they are gone. If your program needs the full reply afterwards, add each piece to a string as it arrives.

In [ ]:
stream = client.chat.completions.create(
  model=MODEL,
  messages=[{"role": "user", "content": "Write a haiku about the ocean."}],
  stream=True
)

haiku = ''
pieces = 0

for chunk in stream:
  if chunk.choices and chunk.choices[0].delta.content:
    piece = chunk.choices[0].delta.content
    print(piece, end='', flush=True)
    haiku += piece
    pieces += 1

print()
print(f'Arrived in {pieces} pieces, {len(haiku)} characters in total.')

# Structured Output

A reply written for a person is awkward for a program. Ask for the coordinates of London and you might get `51.5, -0.13`, or `London is at 51.5°N, 0.13°W`, or a whole paragraph. Your code can't rely on any one of them.

**Structured output** makes the model fill in a form that you design, so the answer always has the same shape.

You describe the form as a class, using a library called **Pydantic**. Each line names a field and gives its type. Then, instead of `create`, you call `parse` and pass your class as `response_format`.

The answer arrives as an object of your class in `message.parsed`. Its fields are real Python values, so `latitude` is a `float` you can do maths with, not text you have to pick apart.

> **Good to know:** Not every model can do structured output, so this section has its own model dropdown.

In [ ]:
from pydantic import BaseModel

STRUCTURED_MODEL = "openai/gpt-5-nano" #@param ["openai/gpt-5-nano", "openai/gpt-5.1-chat", "anthropic/claude-sonnet-5"]

class Location(BaseModel):
  latitude: float
  longitude: float

response = client.chat.completions.parse(
  model=STRUCTURED_MODEL,
  messages=[
    {"role": "system", "content": "You help students with geography."},
    {"role": "user", "content": "What are the GPS coordinates of London?"},
  ],
  response_format=Location
)

location = response.choices[0].message.parsed
print('Latitude:', location.latitude)
print('Longitude:', location.longitude)
print('North of the equator?', location.latitude > 0)

## Lists of Things

A form can hold more than single values. A field can be a list, and the items in the list can be another class of your own.

Below, `Planet` describes one planet, and `PlanetList` is a form with a list of them. The model returns every planet in the same shape, so a `for` loop can go through them, and your code can sort them, count them, or draw them.

In [ ]:
class Planet(BaseModel):
  name: str
  moons: int
  fun_fact: str

class PlanetList(BaseModel):
  planets: list[Planet]

response = client.chat.completions.parse(
  model=STRUCTURED_MODEL,
  messages=[
    {"role": "user", "content": "List the four planets closest to the Sun."},
  ],
  response_format=PlanetList
)

for planet in response.choices[0].message.parsed.planets:
  print(f'{planet.name} (moons: {planet.moons}): {planet.fun_fact}')

# Where to Go Next

You now have the main tools for building with a language model: messages and roles, a conversation that remembers, streaming for long replies, and structured output for answers your code can use.

Some things to try:

- Put the conversation cell in a loop with `input()` to make your own chatbot.
- Design a form for a quiz question, with the question, four options, and the correct answer, then ask the model for a quiz on a topic you choose.
- Stream a story, then read it aloud with [[F3C9A21D-6B4E-4D87-9A0C-5E1F2A3B4C6D|Text to Speech]].
- Run the same code against a model on your own computer in [[3562DE81-27F3-4B25-9E80-EB957E10DD90|Running AI Models Locally]].

# Check Your Understanding